# Retrieval Quality Check

Run the cells below in the `adsp-nlp-backup` environment. The notebook loads `eval_queries.json`, runs the current hybrid retriever, and prints the top 3 chunks for each query.

In [ ]:
from pathlib import Path
import json
import sys
from argparse import Namespace

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "eval_queries.json").exists() and (PROJECT_ROOT.parent / "eval_queries.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from retrieve import retrieve

EVAL_QUERIES_PATH = PROJECT_ROOT / "eval_queries.json"
INDEX_DIR = PROJECT_ROOT / "index"
TOP_K = 3

with EVAL_QUERIES_PATH.open("r", encoding="utf-8") as f:
    eval_queries = json.load(f)

len(eval_queries)

In [ ]:
def make_args(query: str) -> Namespace:
    return Namespace(
        query=query,
        index_dir=str(INDEX_DIR),
        top_k=TOP_K,
        vector_weight=0.50,
        keyword_weight=0.30,
        graph_weight=0.20,
        json=False,
    )


def compact_text(text: str, limit: int = 900) -> str:
    text = " ".join((text or "").split())
    return text if len(text) <= limit else text[:limit].rstrip() + " ..."


def print_query_results(item: dict) -> None:
    qid = item.get("qid", "")
    category = item.get("category", "")
    question = item["question"]
    gold_urls = item.get("gold_urls", [])
    gold_hint = item.get("gold_section_hint", "")

    print("=" * 120)
    print(f"{qid} | {category}")
    print(f"Q: {question}")
    if gold_urls:
        print("Gold URLs:")
        for url in gold_urls:
            print(f"  - {url}")
    else:
        print("Gold URLs: []")
    if gold_hint:
        print(f"Gold hint: {gold_hint}")

    results = retrieve(make_args(question))
    for result in results[:TOP_K]:
        path = " > ".join(result.get("path", []))
        print("-" * 120)
        print(
            f"#{result['rank']} score={result['score']} "
            f"vector={result['vector_score']} keyword={result['keyword_score']} "
            f"graph={result['graph_score']} boost={result['intent_boost']}"
        )
        print(f"Page: {result.get('page_title')}")
        print(f"Path: {path}")
        print(f"URL: {result.get('url')}")
        print(f"Source type: {result.get('source_type')}")
        print("Chunk:")
        print(compact_text(result.get("text", "")))
    print()

In [ ]:
for item in eval_queries:
    print_query_results(item)

Optional: run one query by index if the full output is too long.

In [ ]:
# Change this number to inspect one query at a time.
i = 0
print_query_results(eval_queries[i])